In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
from tqdm.auto import tqdm

In [6]:
df = pd.read_excel(r"C:\Users\Dell\Documents\Elevator BTP\elevator_fault_dataset_enhanced.xlsx")
df.head()

,Timestamp,Motor_Current,Motor_Voltage,Vibration_Level,Speed,Cabin_Position,Cabin_Acceleration,Cabin_Deceleration,Door_Operation_Time,Braking_Force,...,Health_Indices,RUL,Fault_State,Fault_Category,Fault_Severity,Temp_Category,Component_ID_Proxy,High_Speed_Hours,Load_Accumulation,Pulley_Wear_Accumulated
0,2020-01-01 00:00:00,22.289903,241.253222,1.393233,3.519079,19,0.349191,0.358643,5.747832,112.280046,...,171.975594,893.941155,1,Door Failure,Minor,Low (<= 40),1,0,845.186023,84.152230
1,2020-01-01 01:00:00,25.240487,236.078046,0.090303,2.226921,9,0.260127,0.266565,7.552277,142.770141,...,150.250886,2130.455491,0,Brake Failure,Critical,Medium (40-80),1,0,1923.604329,130.980306
2,2020-01-01 02:00:00,32.769471,432.220975,0.932987,2.523211,20,0.291057,0.161099,7.703547,98.016876,...,164.327588,2472.445475,0,Motor Malfunction,Minor,Low (<= 40),1,0,4278.648211,189.880795
3,2020-01-01 03:00:00,34.913962,645.108627,0.490413,4.910428,20,0.439094,0.214058,8.335525,75.588623,...,142.256458,1753.632090,0,Motor Malfunction,Moderate,Medium (40-80),2,0,785.277646,65.480344
4,2020-01-01 04:00:00,27.799073,224.451891,0.844176,5.188226,20,0.566116,0.365294,4.717031,95.882924,...,89.765157,2399.361778,0,Sensor Error,Critical,Medium (40-80),2,0,2796.725017,143.115920


In [3]:
# Preprocess Timestamp
df = df.copy()
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort by component and time
df = df.sort_values(['Component_ID_Proxy', 'Timestamp']).reset_index(drop=True)

In [4]:
# Optional: fill small gaps / missing values sensibly
num_cols = df.select_dtypes(include=['number']).columns.tolist()
# Keep RUL out of imputations
num_cols = [c for c in num_cols if c != 'RUL']

time_cols = [
    'Motor_Current', 'Motor_Voltage', 'Vibration_Level', 'Speed',
    'Cabin_Position', 'Cabin_Acceleration', 'Cabin_Deceleration',
    'Door_Operation_Time', 'Load_Accumulation', 'Pulley_Wear_Accumulated'
]
# Keep only available ones
time_cols = [c for c in time_cols if c in df.columns]

In [5]:
# Define windows (in rows)
WINDOWS = [3, 6, 12, 24, 48, 120, 240]  

roll_features = []

for w in WINDOWS:
    suffix = f'W{w}'
    grp = df.groupby('Component_ID_Proxy')[time_cols]

    # rolling mean/std/min/max
    mean_df = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True).add_suffix(f'_{suffix}_mean')
    std_df  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True).add_suffix(f'_{suffix}_std')
    min_df  = grp.rolling(w, min_periods=1).min().reset_index(level=0, drop=True).add_suffix(f'_{suffix}_min')
    max_df  = grp.rolling(w, min_periods=1).max().reset_index(level=0, drop=True).add_suffix(f'_{suffix}_max')
    last_df = grp.rolling(w, min_periods=1).apply(lambda x: x.iloc[-1]).reset_index(level=0, drop=True).add_suffix(f'_{suffix}_last')

    # simple slope: (last - first) / (window length)
    def slope(arr):
        if len(arr) <= 1:
            return 0.0
        return (arr[-1] - arr[0]) / max(len(arr)-1, 1)
    slope_df = grp.rolling(w, min_periods=1).apply(lambda x: slope(x.values), raw=False).reset_index(level=0, drop=True).add_suffix(f'_{suffix}_slope')

    # concat to dataframe
    df = pd.concat([df, mean_df, std_df, min_df, max_df, last_df, slope_df], axis=1)
    roll_features += list(mean_df.columns) + list(std_df.columns) + list(min_df.columns) + list(max_df.columns) + list(last_df.columns) + list(slope_df.columns)

In [ ]:
cum_cols = ['Load_Accumulation', 'Pulley_Wear_Accumulated', 'High_Speed_Hours']
cum_cols = [c for c in cum_cols if c in df.columns]

for c in cum_cols:
    df[f'{c}_cum'] = df.groupby('Component_ID_Proxy')[c].cumsum()
    roll_features.append(f'{c}_cum')

In [7]:
for c in time_cols:
    df[f'{c}_diff'] = df.groupby('Component_ID_Proxy')[c].diff().fillna(0)
    df[f'{c}_pct_change'] = df.groupby('Component_ID_Proxy')[c].pct_change().fillna(0)
    roll_features += [f'{c}_diff', f'{c}_pct_change']

In [8]:
df['comp_index'] = df.groupby('Component_ID_Proxy').cumcount()  # 0,1,2,...
roll_features.append('comp_index')

df['time_since_start_sec'] = (df['Timestamp'] - df.groupby('Component_ID_Proxy')['Timestamp'].transform('min')).dt.total_seconds()
roll_features.append('time_since_start_sec')

In [9]:
if 'Load' in df.columns and 'Speed' in df.columns:
    df['Load_x_Speed'] = df['Load'] * df['Speed']
    roll_features.append('Load_x_Speed')

if 'Vibration_Level' in df.columns and 'Motor_Current' in df.columns:
    df['Vib_x_Current'] = df['Vibration_Level'] * df['Motor_Current']
    roll_features.append('Vib_x_Current')

In [10]:
static_cols = [
    'Fault_State', 'Fault_Severity', 'Fault_Category', 'Passenger_Count',
    'Battery_Health', 'Number_of_Trips', 'Last_Serviced_Days_Ago', 'MTTF', 'Health_Indices'
]
static_cols = [c for c in static_cols if c in df.columns]

FEATURE_COLS = roll_features + static_cols
FEATURE_COLS = [c for c in FEATURE_COLS if c != 'RUL']  # remove accidental targets
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))        # preserve order, unique

print(f"Number of features: {len(FEATURE_COLS)}")

Number of features: 456


In [11]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = df['Component_ID_Proxy'].values
train_idx, test_idx = next(gss.split(df, groups=groups))

train = df.iloc[train_idx].reset_index(drop=True)
test = df.iloc[test_idx].reset_index(drop=True)

X_train = train[FEATURE_COLS]
y_train = train['RUL']
X_test  = test[FEATURE_COLS]
y_test  = test['RUL']

print("Train rows:", X_train.shape[0], "Test rows:", X_test.shape[0])

Train rows: 34501 Test rows: 8580


In [12]:
# CatBoost can take categorical feature names directly
cat_features = [c for c in static_cols if train[c].dtype == 'object' or train[c].dtype.name == 'category']

for c in cat_features:
    train[c] = train[c].astype('category')
    test[c] = test[c].astype('category')

print("Categorical features for CatBoost:", cat_features)

# Fill any NAs left
X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

# Train CatBoost
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_seed=42,
    verbose=100
)

cat_pool = Pool(X_train, y_train, cat_features=cat_features)
cat_model.fit(cat_pool, eval_set=Pool(X_test, y_test, cat_features=cat_features), use_best_model=True, early_stopping_rounds=50)

# Predictions
y_pred_cat = cat_model.predict(X_test)

print("\n=== CatBoost ===")
print("MAE :", mean_absolute_error(y_test, y_pred_cat))
print("RMSE:", root_mean_squared_error(y_test, y_pred_cat))
print("R²  :", r2_score(y_test, y_pred_cat))

Categorical features for CatBoost: ['Fault_Severity', 'Fault_Category']
0:	learn: 755.9787606	test: 764.8262590	best: 764.8262590 (0)	total: 358ms	remaining: 5m 57s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 764.6598338
bestIteration = 10

Shrink model to first 11 iterations.

=== CatBoost ===
MAE : 630.4414122238934
RMSE: 764.6598340193684
R²  : 0.0003657952417459587


In [13]:
fi = cat_model.get_feature_importance(prettified=True)
print("\nTop CatBoost features:")
print(fi.head(20))


Top CatBoost features:
                          Feature Id  Importances
0              Motor_Voltage_W24_max     5.764288
1            Cabin_Position_W24_mean     3.779246
2         Cabin_Acceleration_W48_max     3.599427
3         Load_Accumulation_W12_mean     2.597966
4       Cabin_Deceleration_W120_last     2.538663
5       Door_Operation_Time_W120_std     2.481754
6            Cabin_Acceleration_diff     2.460077
7           Cabin_Position_W240_last     2.371029
8          Load_Accumulation_W6_last     2.217237
9      Cabin_Acceleration_pct_change     2.064141
10      Door_Operation_Time_W6_slope     1.980840
11                      Speed_W3_max     1.971517
12  Pulley_Wear_Accumulated_W3_slope     1.957754
13            Last_Serviced_Days_Ago     1.954558
14       Cabin_Acceleration_W48_last     1.915646
15  Pulley_Wear_Accumulated_W48_mean     1.904236
16        Load_Accumulation_W240_max     1.881540
17        Load_Accumulation_W48_mean     1.799013
18    Door_Operation_Time_

In [16]:
df.to_excel(r"C:\Users\Dell\Documents\Elevator BTP\elevator_fault_dataset_enhanced_further.xlsx", index=False)